# 01 · B1 TF-IDF baselines

Word (1–2) + character (2–5) n-gram TF-IDF with logistic regression and a linear SVM. C is selected on the validation split by macro-F1; the test split is used once. CPU is enough (~2 min).

In [1]:
# ==== Setup (run first) ====
# SOURCE = "drive"  : the repository folder is on Google Drive (use this now)
# SOURCE = "github" : clone from GitHub (after the repository is public)
SOURCE = "drive"
DRIVE_REPO = "/content/drive/MyDrive/2026/Publikasi Sinta 2 Baru/mad-indo-sentiment"   # folder from the unzipped package
REPO_URL = "https://github.com/deepittalk-repo/mad-indo-sentiment.git"   # used only when SOURCE = "github"

import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")          # results persist in MyDrive/mad-indo-results
    if SOURCE == "drive":
        REPO_DIR = DRIVE_REPO
        assert os.path.exists(REPO_DIR), f"Folder not found: {REPO_DIR}. Upload the unzipped folder to MyDrive first."
    else:
        REPO_DIR = "/content/mad-indo-sentiment"
        if not os.path.exists(REPO_DIR):
            !git clone -q $REPO_URL $REPO_DIR
    %cd $REPO_DIR
    !pip -q install -r requirements.txt
else:
    # Local run: start Jupyter from the repository root or the notebooks/ folder.
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
sys.path.insert(0, "src")

from mad_indo import data
cfg = data.load_config()
RESULTS = data.results_dir()
print("Results folder:", RESULTS)
LIMIT = cfg["limit_test_samples"]   # null in config = full test set


Mounted at /content/drive
/content/drive/MyDrive/2026/Publikasi Sinta 2 Baru/mad-indo-sentiment
Results folder: /content/drive/MyDrive/mad-indo-results


In [2]:
import json, pandas as pd
from mad_indo import baselines, stats
out = RESULTS / "baselines"; out.mkdir(exist_ok=True)
summary = []
for task in ["smsa", "emot"]:
    L = cfg["tasks"][task]["labels"]
    tr, va = data.load_split(task, "train"), data.load_split(task, "valid")
    te = data.load_split(task, "test", limit=LIMIT)
    res = baselines.run_tfidf(tr, va, te, L, cfg["baselines"]["tfidf"]["C_grid"])
    for name, r in res.items():
        pd.DataFrame({"id": te.id, "gold": te.label, "pred": r["pred"]}).to_csv(out / f"{name}_{task}.csv", index=False)
        json.dump({k: v for k, v in r.items() if k != "pred"}, open(out / f"{name}_{task}.json", "w"), indent=2)
        summary.append({"task": task, "method": name, "best_C": r["best_C"], **stats.metrics(te.label, r["pred"], L)})
pd.DataFrame(summary).round(4)

/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


,task,method,best_C,accuracy,macro_f1,unknown_rate,f1_positive,recall_positive,f1_neutral,recall_neutral,...,f1_sadness,recall_sadness,f1_anger,recall_anger,f1_love,recall_love,f1_fear,recall_fear,f1_happy,recall_happy
0,smsa,B1_tfidf_logreg,5.0,0.8060,0.7505,0.0,0.8479,0.8173,0.5630,0.4318,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,smsa,B1_tfidf_linsvm,0.5,0.8060,0.7520,0.0,0.8492,0.8125,0.5634,0.4545,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,emot,B1_tfidf_logreg,5.0,0.6886,0.6992,0.0,NaN,NaN,NaN,NaN,...,0.6311,0.65,0.7049,0.7818,0.7937,0.7812,0.7273,0.6154,0.6392,0.6139
3,emot,B1_tfidf_linsvm,0.1,0.6659,0.6744,0.0,NaN,NaN,NaN,NaN,...,0.5683,0.52,0.6873,0.8091,0.7669,0.7969,0.7207,0.6154,0.6289,0.6040
